<a href="https://colab.research.google.com/github/SSS-LEARN/FDP-AGENENTIC-AI-RAG/blob/main/PHD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!mkdir -p /content/drive/MyDrive/sen1floods11/HandLabeled
!ls /content/drive/MyDrive/sen1floods11

HandLabeled


In [ ]:
!ls /content/drive/MyDrive/sen1floods11/HandLabeled
!du -sh /content/drive/MyDrive/sen1floods11/HandLabeled

4.0K	/content/drive/MyDrive/sen1floods11/HandLabeled


In [ ]:
!pip install -q segmentation-models-pytorch rasterio torchmetrics

In [ ]:
!gsutil ls gs://sen1floods11/v1.1/data/flood_events/HandLabeled/

Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
gs://sen1floods11/v1.1/data/flood_events/HandLabeled/JRCWaterHand/
gs://sen1floods11/v1.1/data/flood_events/HandLabeled/LabelHand/
gs://sen1floods11/v1.1/data/flood_events/HandLabeled/S1Hand/
gs://sen1floods11/v1.1/data/flood_events/HandLabeled/S1OtsuLabelHand/
gs://sen1floods11/v1.1/data/flood_events/HandLabeled/S2Hand/


In [ ]:
!gsutil -m rsync -r gs://sen1floods11/v1.1/data/flood_events/HandLabeled /content/drive/MyDrive/sen1floods11/HandLabeled/

Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.

both the source and destination. Your crcmod installation isn't using the
module's C extension, so checksumming will run very slowly. If this is your
first rsync since updating gsutil, this rsync can take significantly longer than
usual. For help installing the extension, please see "gsutil help crcmod".

Building synchronization state...
Starting synchronization...
Copying gs://sen1floods11/v1.1/data/flood_events/HandLabeled/JRCWaterHand/Bolivia_290290_JRCWaterHand.tif...
Copying gs://sen1floods11/v1.1/data/flood_events/HandLabeled/JRCWaterHand/Bolivia_129334_JRCWaterHand.tif...
Copying gs://sen1floods11/v1.1/data/flood_events/HandLabeled/JRCWaterHand/Bolivia_195474_JRCWaterHand.tif...
Copying gs://sen1floods11/v1.1/data/flood_ev

In [ ]:
!du -sh /content/drive/MyDrive/sen1floods11/HandLabeled
!ls /content/drive/MyDrive/sen1floods11/HandLabeled


1.7G	/content/drive/MyDrive/sen1floods11/HandLabeled
JRCWaterHand  LabelHand  S1Hand  S1OtsuLabelHand  S2Hand


In [ ]:
!pip install -q segmentation-models-pytorch rasterio torchmetrics

In [ ]:
!ls /content/drive/MyDrive/sen1floods11/HandLabeled

JRCWaterHand  LabelHand  S1Hand  S1OtsuLabelHand  S2Hand


In [ ]:
import torch
print(torch.cuda.is_available())

True


In [ ]:
import os, glob
import numpy as np
import rasterio
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
import segmentation_models_pytorch as smp
from torchmetrics.classification import BinaryJaccardIndex

# --- paths: pointed at the data you already downloaded ---
S1_DIR = "/content/drive/MyDrive/sen1floods11/HandLabeled/S1Hand"
LABEL_DIR = "/content/drive/MyDrive/sen1floods11/HandLabeled/LabelHand"
BATCH_SIZE = 8
NUM_EPOCHS = 30
LR = 1e-4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CHECKPOINT_PATH = "/content/drive/MyDrive/sen1floods11/flood_unet_baseline.pt"

print("Using device:", DEVICE)

class Sen1Floods11Dataset(Dataset):
    def __init__(self, s1_dir, label_dir):
        self.s1_paths = sorted(glob.glob(os.path.join(s1_dir, "*_S1Hand.tif")))
        self.label_dir = label_dir
        if len(self.s1_paths) == 0:
            raise FileNotFoundError(f"No S1 chips found in {s1_dir}. Check Cell B's output and fix the path above.")

    def __len__(self):
        return len(self.s1_paths)

    def __getitem__(self, idx):
        s1_path = self.s1_paths[idx]
        chip_id = os.path.basename(s1_path).replace("_S1Hand.tif", "")
        label_path = os.path.join(self.label_dir, f"{chip_id}_LabelHand.tif")

        with rasterio.open(s1_path) as src:
            s1 = src.read().astype(np.float32)
        with rasterio.open(label_path) as src:
            label = src.read(1).astype(np.float32)

        s1 = np.nan_to_num(s1, nan=-50.0, posinf=1.0, neginf=-50.0)  # <-- NEW: clean NaN/Inf before clipping
        s1 = np.clip(s1, -50, 1)
        s1 = (s1 + 50) / 51.0
        label = np.where(label == -1, 255, label)

        return torch.from_numpy(s1), torch.from_numpy(label).long()

def build_model():
    return smp.Unet(encoder_name="resnet34", encoder_weights="imagenet", in_channels=2, classes=1)

def masked_bce_loss(logits, targets):
    valid = (targets != 255)
    targets_f = targets.float()
    loss = nn.functional.binary_cross_entropy_with_logits(logits.squeeze(1), targets_f, reduction="none")
    return loss[valid].mean()

dataset = Sen1Floods11Dataset(S1_DIR, LABEL_DIR)
n_val = max(1, int(0.15 * len(dataset)))
n_train = len(dataset) - n_val
train_ds, val_ds = random_split(dataset, [n_train, n_val])
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

model = build_model().to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
iou_metric = BinaryJaccardIndex(ignore_index=255).to(DEVICE)

best_iou = 0.0
for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    train_loss = 0.0
    for s1, label in train_loader:
        s1, label = s1.to(DEVICE), label.to(DEVICE)
        optimizer.zero_grad()
        logits = model(s1)
        loss = masked_bce_loss(logits, label)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * s1.size(0)
    train_loss /= len(train_ds)

    model.eval()
    iou_metric.reset()
    with torch.no_grad():
        for s1, label in val_loader:
            s1, label = s1.to(DEVICE), label.to(DEVICE)
            logits = model(s1)
            preds = (torch.sigmoid(logits.squeeze(1)) > 0.5).long()
            iou_metric.update(preds, label)
    val_iou = iou_metric.compute().item()

    print(f"Epoch {epoch:02d}/{NUM_EPOCHS} | train_loss={train_loss:.4f} | val_IoU={val_iou:.4f}")

    if val_iou > best_iou:
        best_iou = val_iou
        torch.save(model.state_dict(), CHECKPOINT_PATH)
        print(f"  -> new best model saved (IoU={best_iou:.4f})")

print(f"\nTraining complete. Best val IoU: {best_iou:.4f}")
print("Sanity check: published SAR-only baselines on this set are typically ~0.55-0.65 IoU.")

Using device: cuda


config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 87.3MB            

model.safetensors: downloading bytes:           |  0.00B            

Epoch 01/30 | train_loss=0.4850 | val_IoU=0.5831
  -> new best model saved (IoU=0.5831)
Epoch 02/30 | train_loss=0.3443 | val_IoU=0.6225
  -> new best model saved (IoU=0.6225)


KeyboardInterrupt: 

In [ ]:
import rasterio, glob

sample_s2 = glob.glob("/content/drive/MyDrive/sen1floods11/HandLabeled/S2Hand/*.tif")[0]
with rasterio.open(sample_s2) as src:
    print("S2 file:", sample_s2)
    print("Number of bands:", src.count)

S2 file: /content/drive/MyDrive/sen1floods11/HandLabeled/S2Hand/Paraguay_148318_S2Hand.tif
Number of bands: 13


In [ ]:
!ls /content/drive/MyDrive/sen1floods11/HandLabeled
print("---")
!ls /content/drive/MyDrive/sen1floods11/HandLabeled/S2Hand | head -5
print("---")
!du -sh /content/drive/MyDrive/sen1floods11/HandLabeled/S2Hand

JRCWaterHand  LabelHand  S1Hand  S1OtsuLabelHand  S2Hand
---
Bolivia_103757_S2Hand.tif
Bolivia_129334_S2Hand.tif
Bolivia_195474_S2Hand.tif
Bolivia_23014_S2Hand.tif
Bolivia_233925_S2Hand.tif
---
972M	/content/drive/MyDrive/sen1floods11/HandLabeled/S2Hand


In [ ]:
import os, glob
import numpy as np
import rasterio
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
import segmentation_models_pytorch as smp
from torchmetrics.classification import BinaryJaccardIndex

S1_DIR = "/content/drive/MyDrive/sen1floods11/HandLabeled/S1Hand"
S2_DIR = "/content/drive/MyDrive/sen1floods11/HandLabeled/S2Hand"
LABEL_DIR = "/content/drive/MyDrive/sen1floods11/HandLabeled/LabelHand"
BATCH_SIZE = 8
NUM_EPOCHS = 30
LR = 1e-4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CHECKPOINT_PATH = "/content/drive/MyDrive/sen1floods11/flood_unet_multimodal.pt"

print("Using device:", DEVICE)

class Sen1Floods11MultimodalDataset(Dataset):
    def __init__(self, s1_dir, s2_dir, label_dir):
        all_s1 = sorted(glob.glob(os.path.join(s1_dir, "*_S1Hand.tif")))
        # keep only chips that have a matching S2 file too
        self.chip_ids = []
        for p in all_s1:
            chip_id = os.path.basename(p).replace("_S1Hand.tif", "")
            s2_path = os.path.join(s2_dir, f"{chip_id}_S2Hand.tif")
            if os.path.exists(s2_path):
                self.chip_ids.append(chip_id)
        self.s1_dir, self.s2_dir, self.label_dir = s1_dir, s2_dir, label_dir
        if len(self.chip_ids) == 0:
            raise FileNotFoundError("No chips with matching S1+S2 pairs found — check folder paths.")
        print(f"Found {len(self.chip_ids)} chips with matching SAR+optical pairs "
              f"(out of {len(all_s1)} total SAR chips).")

    def __len__(self):
        return len(self.chip_ids)

    def __getitem__(self, idx):
        chip_id = self.chip_ids[idx]
        s1_path = os.path.join(self.s1_dir, f"{chip_id}_S1Hand.tif")
        s2_path = os.path.join(self.s2_dir, f"{chip_id}_S2Hand.tif")
        label_path = os.path.join(self.label_dir, f"{chip_id}_LabelHand.tif")

        with rasterio.open(s1_path) as src:
            s1 = src.read().astype(np.float32)
        with rasterio.open(s2_path) as src:
            s2 = src.read().astype(np.float32)
        with rasterio.open(label_path) as src:
            label = src.read(1).astype(np.float32)

        # SAR: clean, clip, normalize to ~0-1
        s1 = np.nan_to_num(s1, nan=-50.0, posinf=1.0, neginf=-50.0)
        s1 = np.clip(s1, -50, 1)
        s1 = (s1 + 50) / 51.0

        # Optical: clean, clip reflectance DN range, normalize to ~0-1
        s2 = np.nan_to_num(s2, nan=0.0, posinf=10000.0, neginf=0.0)
        s2 = np.clip(s2, 0, 10000) / 10000.0

        stacked = np.concatenate([s1, s2], axis=0)  # (2+13, H, W) = (15, H, W)
        label = np.where(label == -1, 255, label)

        return torch.from_numpy(stacked), torch.from_numpy(label).long()

def build_model(in_channels):
    return smp.Unet(encoder_name="resnet34", encoder_weights="imagenet", in_channels=in_channels, classes=1)

def masked_bce_loss(logits, targets):
    valid = (targets != 255)
    targets_f = targets.float()
    loss = nn.functional.binary_cross_entropy_with_logits(logits.squeeze(1), targets_f, reduction="none")
    return loss[valid].mean()

dataset = Sen1Floods11MultimodalDataset(S1_DIR, S2_DIR, LABEL_DIR)
n_val = max(1, int(0.15 * len(dataset)))
n_train = len(dataset) - n_val
train_ds, val_ds = random_split(dataset, [n_train, n_val])
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

model = build_model(in_channels=15).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
iou_metric = BinaryJaccardIndex(ignore_index=255).to(DEVICE)

best_iou = 0.0
for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    train_loss = 0.0
    for x, label in train_loader:
        x, label = x.to(DEVICE), label.to(DEVICE)
        optimizer.zero_grad()
        logits = model(x)
        loss = masked_bce_loss(logits, label)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * x.size(0)
    train_loss /= len(train_ds)

    model.eval()
    iou_metric.reset()
    with torch.no_grad():
        for x, label in val_loader:
            x, label = x.to(DEVICE), label.to(DEVICE)
            logits = model(x)
            preds = (torch.sigmoid(logits.squeeze(1)) > 0.5).long()
            iou_metric.update(preds, label)
    val_iou = iou_metric.compute().item()

    print(f"Epoch {epoch:02d}/{NUM_EPOCHS} | train_loss={train_loss:.4f} | val_IoU={val_iou:.4f}")

    if val_iou > best_iou:
        best_iou = val_iou
        torch.save(model.state_dict(), CHECKPOINT_PATH)
        print(f"  -> new best model saved (IoU={best_iou:.4f})")

print(f"\nMultimodal training complete. Best val IoU: {best_iou:.4f}")
print(f"Compare against SAR-only baseline: 0.6421")

Using device: cuda
Found 446 chips with matching SAR+optical pairs (out of 446 total SAR chips).


config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 87.3MB            

model.safetensors: downloading bytes:           |  0.00B            

Epoch 01/30 | train_loss=0.6675 | val_IoU=0.5042
  -> new best model saved (IoU=0.5042)
Epoch 02/30 | train_loss=0.4713 | val_IoU=0.6333
  -> new best model saved (IoU=0.6333)
Epoch 03/30 | train_loss=0.3852 | val_IoU=0.6324
Epoch 04/30 | train_loss=0.3396 | val_IoU=0.4783
Epoch 05/30 | train_loss=0.2938 | val_IoU=0.7587
  -> new best model saved (IoU=0.7587)
Epoch 06/30 | train_loss=0.2611 | val_IoU=0.7101
Epoch 07/30 | train_loss=0.2295 | val_IoU=0.7660
  -> new best model saved (IoU=0.7660)
Epoch 08/30 | train_loss=0.2153 | val_IoU=0.7725
  -> new best model saved (IoU=0.7725)
Epoch 09/30 | train_loss=0.1891 | val_IoU=0.7306
Epoch 10/30 | train_loss=0.1841 | val_IoU=0.7418
Epoch 11/30 | train_loss=0.1673 | val_IoU=0.7990
  -> new best model saved (IoU=0.7990)
Epoch 12/30 | train_loss=0.1529 | val_IoU=0.7829
Epoch 13/30 | train_loss=0.1506 | val_IoU=0.7751
Epoch 14/30 | train_loss=0.1428 | val_IoU=0.7858
Epoch 15/30 | train_loss=0.1299 | val_IoU=0.7694
Epoch 16/30 | train_loss=0.1238

In [ ]:
!find / -name "flood_unet_multimodal.pt" 2>/dev/null

/content/drive/MyDrive/sen1floods11/flood_unet_multimodal.pt


In [ ]:
import torch
import numpy as np
from torchmetrics.classification import BinaryJaccardIndex

# --- Load your trained multimodal model ---
device = "cuda" if torch.cuda.is_available() else "cpu"
model.load_state_dict(torch.load("/content/drive/MyDrive/sen1floods11/flood_unet_multimodal.pt", map_location=device))
model.eval()

# --- IMPORTANT: confirm channel order ---
# This assumes your 15-channel input is [SAR_VV, SAR_VH, optical_band_1, ..., optical_band_13]
# i.e. channels 0-1 = SAR, channels 2-14 = optical.
# If your code built the tensor in a different order, change SAR_CHANNELS below.
SAR_CHANNELS = 2
TOTAL_CHANNELS = 15

def simulate_cloud_cover(img, coverage_frac, rng):
    """
    img: numpy array, shape (channels, H, W)
    coverage_frac: fraction of the image area to block out in optical channels (0.0 to 1.0)
    Zeros out a random rectangular patch covering roughly coverage_frac of the image,
    only in the optical channels (SAR channels untouched).
    """
    img = img.copy()
    if coverage_frac <= 0:
        return img
    C, H, W = img.shape
    # random rectangle with area ~ coverage_frac * H * W
    patch_h = int(H * np.sqrt(coverage_frac))
    patch_w = int(W * np.sqrt(coverage_frac))
    patch_h = min(patch_h, H)
    patch_w = min(patch_w, W)
    top = rng.integers(0, max(1, H - patch_h + 1))
    left = rng.integers(0, max(1, W - patch_w + 1))
    img[SAR_CHANNELS:TOTAL_CHANNELS, top:top+patch_h, left:left+patch_w] = 0.0
    return img

# --- Run evaluation at several cloud-cover levels ---
coverage_levels = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
jaccard = BinaryJaccardIndex(ignore_index=255).to(device)
rng = np.random.default_rng(42)

results = {}
for cov in coverage_levels:
    jaccard.reset()
    with torch.no_grad():
        for img, mask in val_loader:   # use your existing validation DataLoader
            img_np = img.numpy()
            img_clouded = np.stack([simulate_cloud_cover(im, cov, rng) for im in img_np])
            img_clouded = torch.from_numpy(img_clouded).float().to(device)
            mask = mask.to(device)
            logits = model(img_clouded)
            preds = (torch.sigmoid(logits.squeeze(1)) > 0.5).long()
            jaccard.update(preds, mask.long())
    iou = jaccard.compute().item()
    results[cov] = iou
    print(f"Cloud cover {int(cov*100)}%: IoU = {iou:.4f}")

print("\nSummary:", results)
print(f"\nFor reference, SAR-only baseline (no optical at all) achieved IoU = 0.6421")

Cloud cover 0%: IoU = 0.8215
Cloud cover 20%: IoU = 0.6157
Cloud cover 40%: IoU = 0.4723
Cloud cover 60%: IoU = 0.3797
Cloud cover 80%: IoU = 0.3135
Cloud cover 100%: IoU = 0.2320

Summary: {0.0: 0.8215479850769043, 0.2: 0.6157438158988953, 0.4: 0.4723324477672577, 0.6: 0.37974950671195984, 0.8: 0.31352314352989197, 1.0: 0.23197245597839355}

For reference, SAR-only baseline (no optical at all) achieved IoU = 0.6421


In [ ]:
import os
os.makedirs("/content/drive/MyDrive/landslide4sense", exist_ok=True)

!wget -q --show-progress -O /content/drive/MyDrive/landslide4sense/TrainData.zip "https://zenodo.org/records/10463239/files/TrainData.zip?download=1"
!wget -q --show-progress -O /content/drive/MyDrive/landslide4sense/ValidData.zip "https://zenodo.org/records/10463239/files/ValidData.zip?download=1"
!wget -q --show-progress -O /content/drive/MyDrive/landslide4sense/TestData.zip "https://zenodo.org/records/10463239/files/TestData.zip?download=1"

/content/drive/MyDr 100%[===================>]   2.31G  1.80MB/s    in 18m 26s 
/content/drive/MyDr 100%[===================>] 136.78M  2.50MB/s    in 87s     
/content/drive/MyDr 100%[===================>] 449.08M  1.73MB/s    in 3m 43s  


In [ ]:
!unzip -q /content/drive/MyDrive/landslide4sense/TrainData.zip -d /content/drive/MyDrive/landslide4sense/
!unzip -q /content/drive/MyDrive/landslide4sense/ValidData.zip -d /content/drive/MyDrive/landslide4sense/
!unzip -q /content/drive/MyDrive/landslide4sense/TestData.zip -d /content/drive/MyDrive/landslide4sense/

In [ ]:
!ls /content/drive/MyDrive/landslide4sense
!du -sh /content/drive/MyDrive/landslide4sense/TrainData

TestData  TestData.zip	TrainData  TrainData.zip  ValidData  ValidData.zip
6.6G	/content/drive/MyDrive/landslide4sense/TrainData


In [ ]:
import h5py
import numpy as np

img_path = "/content/drive/MyDrive/landslide4sense/TrainData/img/image_1.h5"
mask_path = "/content/drive/MyDrive/landslide4sense/TrainData/mask/mask_1.h5"

with h5py.File(img_path, "r") as f:
    key = list(f.keys())[0]
    img = f[key][:]
    print("Image key:", key, "| shape:", img.shape, "| dtype:", img.dtype)
    print("Image min/max:", img.min(), img.max())

with h5py.File(mask_path, "r") as f:
    key = list(f.keys())[0]
    mask = f[key][:]
    print("Mask key:", key, "| shape:", mask.shape, "| dtype:", mask.dtype)
    print("Unique mask values:", np.unique(mask))


Image key: img | shape: (128, 128, 14) | dtype: float64
Image min/max: 0.0 6.397335147428777
Mask key: mask | shape: (128, 128) | dtype: uint8
Unique mask values: [0 1]


In [ ]:
import os, re, glob
import numpy as np
import h5py
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
import segmentation_models_pytorch as smp
from torchmetrics.classification import BinaryJaccardIndex

IMG_DIR = "/content/drive/MyDrive/landslide4sense/TrainData/img"
MASK_DIR = "/content/drive/MyDrive/landslide4sense/TrainData/mask"
BATCH_SIZE = 16
NUM_EPOCHS = 30
LR = 1e-4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CHECKPOINT_PATH = "/content/drive/MyDrive/landslide4sense/landslide_unet_baseline.pt"

print("Using device:", DEVICE)

class Landslide4SenseDataset(Dataset):
    def __init__(self, img_dir, mask_dir):
        self.img_paths = sorted(
            glob.glob(os.path.join(img_dir, "image_*.h5")),
            key=lambda p: int(re.search(r"image_(\d+)\.h5", p).group(1))
        )
        self.mask_dir = mask_dir
        if len(self.img_paths) == 0:
            raise FileNotFoundError(f"No files found in {img_dir}")
        print(f"Found {len(self.img_paths)} image/mask pairs.")

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        img_path = self.img_paths[idx]
        n = re.search(r"image_(\d+)\.h5", img_path).group(1)
        mask_path = os.path.join(self.mask_dir, f"mask_{n}.h5")

        with h5py.File(img_path, "r") as f:
            img = f[list(f.keys())[0]][:].astype(np.float32)  # (128, 128, 14)
        with h5py.File(mask_path, "r") as f:
            mask = f[list(f.keys())[0]][:].astype(np.float32)  # (128, 128)

        img = np.nan_to_num(img, nan=0.0, posinf=5.0, neginf=0.0)
        img = np.clip(img, 0, 5) / 5.0
        img = np.transpose(img, (2, 0, 1))  # -> (14, 128, 128) for PyTorch

        return torch.from_numpy(img), torch.from_numpy(mask).long()

def build_model(in_channels):
    return smp.Unet(encoder_name="resnet34", encoder_weights="imagenet", in_channels=in_channels, classes=1)

def bce_loss(logits, targets):
    return nn.functional.binary_cross_entropy_with_logits(logits.squeeze(1), targets.float())

dataset = Landslide4SenseDataset(IMG_DIR, MASK_DIR)
n_val = max(1, int(0.15 * len(dataset)))
n_train = len(dataset) - n_val
train_ds, val_ds = random_split(dataset, [n_train, n_val])
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

model = build_model(in_channels=14).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
iou_metric = BinaryJaccardIndex().to(DEVICE)

best_iou = 0.0
for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    train_loss = 0.0
    for x, mask in train_loader:
        x, mask = x.to(DEVICE), mask.to(DEVICE)
        optimizer.zero_grad()
        logits = model(x)
        loss = bce_loss(logits, mask)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * x.size(0)
    train_loss /= len(train_ds)

    model.eval()
    iou_metric.reset()
    with torch.no_grad():
        for x, mask in val_loader:
            x, mask = x.to(DEVICE), mask.to(DEVICE)
            logits = model(x)
            preds = (torch.sigmoid(logits.squeeze(1)) > 0.5).long()
            iou_metric.update(preds, mask)
    val_iou = iou_metric.compute().item()

    print(f"Epoch {epoch:02d}/{NUM_EPOCHS} | train_loss={train_loss:.4f} | val_IoU={val_iou:.4f}")

    if val_iou > best_iou:
        best_iou = val_iou
        torch.save(model.state_dict(), CHECKPOINT_PATH)
        print(f"  -> new best model saved (IoU={best_iou:.4f})")

print(f"\nLandslide baseline training complete. Best val IoU: {best_iou:.4f}")

Using device: cuda
Found 3799 image/mask pairs.


config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 87.3MB            

model.safetensors: downloading bytes:           |  0.00B            

Epoch 01/30 | train_loss=0.3639 | val_IoU=0.2278
  -> new best model saved (IoU=0.2278)
Epoch 02/30 | train_loss=0.1754 | val_IoU=0.2400
  -> new best model saved (IoU=0.2400)
Epoch 03/30 | train_loss=0.1104 | val_IoU=0.3604
  -> new best model saved (IoU=0.3604)
Epoch 04/30 | train_loss=0.0796 | val_IoU=0.4261
  -> new best model saved (IoU=0.4261)
Epoch 05/30 | train_loss=0.0630 | val_IoU=0.4226
Epoch 06/30 | train_loss=0.0524 | val_IoU=0.5465
  -> new best model saved (IoU=0.5465)
Epoch 07/30 | train_loss=0.0453 | val_IoU=0.4682
Epoch 08/30 | train_loss=0.0410 | val_IoU=0.5592
  -> new best model saved (IoU=0.5592)
Epoch 09/30 | train_loss=0.0383 | val_IoU=0.5585
Epoch 10/30 | train_loss=0.0363 | val_IoU=0.5580
Epoch 11/30 | train_loss=0.0340 | val_IoU=0.5559
Epoch 12/30 | train_loss=0.0321 | val_IoU=0.5599
  -> new best model saved (IoU=0.5599)
Epoch 13/30 | train_loss=0.0304 | val_IoU=0.5037
Epoch 14/30 | train_loss=0.0301 | val_IoU=0.5618
  -> new best model saved (IoU=0.5618)
Ep